# Breast Tumor Classification: PyTorch vs. Linear SVM
**Goal:** Reproduce the scikit-learn tumor classifier (PCA + linear SVM) with a PyTorch model, using the same Wisconsin Breast Cancer data and a leakage-free pipeline, and compare the two honestly.

**Pipeline:** ingest, brief EDA, stratified train/val/test split, scaling + PCA (fit on train only), `Dataset`/`DataLoader`, MLP with early stopping, multi-seed evaluation against an SVM baseline.

**Label convention:** `0 = Malignant`, `1 = Benign` (scikit-learn's encoding). Malignant is treated as the clinically important class, so *sensitivity* = malignant recall and *specificity* = benign recall.

In [ ]:
import copy
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

SPLIT_SEED = 24
N_COMPONENTS = 10   # PCs fed to the models; the first 2 are also used for plotting
N_SEEDS = 5         # repeat training to report mean +/- std

## 1. Data Ingestion

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

df = pd.DataFrame(X, columns=data.feature_names)
df['Diagnosis'] = pd.Series(y).map({0: 'Malignant', 1: 'Benign'})
logging.info(f"Loaded {df.shape[0]} samples, {X.shape[1]} features")

## 2. Brief EDA

In [ ]:
print(f"Missing values: {df.isnull().sum().sum()}")
print(df['Diagnosis'].value_counts(normalize=True).mul(100).round(2).to_string())
display(df.drop(columns='Diagnosis').describe().T[['mean', 'std', 'min', 'max']].head(10))

The classes are moderately imbalanced (about 63% benign / 37% malignant), so every split is stratified. Feature scales differ by orders of magnitude (e.g. `mean area` vs. `mean smoothness`), so standardization must happen before PCA.

## 3. Split, Scale, and PCA (fit on training data only)
Train/val/test = 64/16/20. The validation set is used only for early stopping. The test set is touched once for final evaluation.

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SPLIT_SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.2, stratify=y_tmp, random_state=SPLIT_SEED)

scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=N_COMPONENTS).fit(scaler.transform(X_train))
prep = lambda A: pca.transform(scaler.transform(A)).astype(np.float32)

Z_train, Z_val, Z_test = prep(X_train), prep(X_val), prep(X_test)

print(f"Train/Val/Test: {len(y_train)}/{len(y_val)}/{len(y_test)}")
print(f"Variance retained: 2 PCs = {pca.explained_variance_ratio_[:2].sum()*100:.1f}%, "
      f"{N_COMPONENTS} PCs = {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
plot_df = pd.DataFrame(Z_train[:, :2], columns=['PC1', 'PC2'])
plot_df['Diagnosis'] = pd.Series(y_train).map({0: 'Malignant', 1: 'Benign'})

plt.figure(figsize=(8, 5))
sb.scatterplot(data=plot_df, x='PC1', y='PC2', hue='Diagnosis', alpha=0.8)
plt.title("Training set in PCA space")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.grid(True)
plt.show()

## 4. PyTorch Data Pipeline
`TensorDataset` + `DataLoader` handle batching and shuffling. This is the same pattern that scales to larger datasets like TCGA.

In [ ]:
def make_loader(Z, labels, shuffle):
    ds = TensorDataset(torch.from_numpy(Z), torch.from_numpy(labels.astype(np.float32)))
    return DataLoader(ds, batch_size=32, shuffle=shuffle)

train_dl = make_loader(Z_train, y_train, shuffle=True)

## 5. Model, Training Loop, and Early Stopping
A small MLP outputs a single raw logit, and `BCEWithLogitsLoss` applies the sigmoid internally, which is numerically stable. The best weights by validation loss are restored at the end.

In [ ]:
def make_mlp(in_dim):
    return nn.Sequential(
        nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(32, 16), nn.ReLU(),
        nn.Linear(16, 1),
    )

loss_fn = nn.BCEWithLogitsLoss()
Zv, yv = torch.from_numpy(Z_val), torch.from_numpy(y_val.astype(np.float32))


@torch.no_grad()
def predict_logits(model, Z):
    model.eval()
    return model(torch.from_numpy(Z)).squeeze(1).numpy()


def train_mlp(seed, max_epochs=300, patience=20, lr=1e-3, weight_decay=1e-4):
    torch.manual_seed(seed)
    model = make_mlp(N_COMPONENTS)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_loss, best_state, bad, history = float('inf'), None, 0, []

    for epoch in range(max_epochs):
        model.train()
        running = 0.0
        for xb, yb in train_dl:
            opt.zero_grad()
            loss = loss_fn(model(xb).squeeze(1), yb)
            loss.backward()
            opt.step()
            running += loss.item() * len(xb)

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Zv).squeeze(1), yv).item()
        history.append((running / len(Z_train), val_loss))

        if val_loss < best_loss:
            best_loss, best_state, bad = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return model, history

## 6. Evaluation Helpers

In [ ]:
def compute_metrics(y_true, y_pred, y_score):
    malignant, benign = (y_true == 0), (y_true == 1)
    return {
        'accuracy': (y_pred == y_true).mean(),
        'sensitivity (malignant recall)': (y_pred[malignant] == 0).mean(),
        'specificity (benign recall)': (y_pred[benign] == 1).mean(),
        'AUC': roc_auc_score(y_true, y_score),
    }

## 7. Train and Evaluate
The SVM baselines train on the same 64% training split. The 2-PC SVM mirrors the original notebook's setup. The PyTorch MLP is trained with several seeds because neural-net results vary run to run.

In [ ]:
rows = {}

# SVM baselines (deterministic)
for k in (2, N_COMPONENTS):
    svm = SVC(kernel='linear', C=1.0, random_state=42).fit(Z_train[:, :k], y_train)
    pred = svm.predict(Z_test[:, :k])
    score = svm.decision_function(Z_test[:, :k])
    rows[f'Linear SVM ({k} PCs)'] = {m: [v] for m, v in compute_metrics(y_test, pred, score).items()}

# PyTorch MLP across seeds
mlp_runs, models, histories = [], [], []
for seed in range(N_SEEDS):
    model, hist = train_mlp(seed)
    logits = predict_logits(model, Z_test)
    pred = (logits > 0).astype(int)
    mlp_runs.append(compute_metrics(y_test, pred, logits))
    models.append(model); histories.append(hist)
    logging.info(f"seed {seed}: {len(hist)} epochs, test acc {mlp_runs[-1]['accuracy']:.4f}")

rows[f'PyTorch MLP ({N_COMPONENTS} PCs)'] = {
    m: [r[m] for r in mlp_runs] for m in mlp_runs[0]}

summary = pd.DataFrame({
    name: {m: f"{np.mean(v)*100:.2f}" + (f" +/- {np.std(v)*100:.2f}" if len(v) > 1 else "")
           if m != 'AUC' else f"{np.mean(v):.3f}" + (f" +/- {np.std(v):.3f}" if len(v) > 1 else "")
           for m, v in metrics.items()}
    for name, metrics in rows.items()
}).T
display(summary)

### Training curves and detailed report (seed 0)

In [ ]:
hist = np.array(histories[0])
plt.figure(figsize=(7, 4))
plt.plot(hist[:, 0], label='train loss')
plt.plot(hist[:, 1], label='val loss')
plt.xlabel('epoch'); plt.ylabel('BCE loss'); plt.legend(); plt.grid(True)
plt.title("MLP training curves (seed 0)")
plt.show()

logits0 = predict_logits(models[0], Z_test)
print(classification_report(y_test, (logits0 > 0).astype(int),
                            target_names=['Malignant', 'Benign']))

torch.save(models[0].state_dict(), 'tumor_mlp_seed0.pt')

## 8. Interpretation
*Fill in after running:*
- How does the 2-PC SVM compare to the original notebook's 95.61% (different split sizes, so small differences are expected)?
- Does the 10-PC input help either model?
- Does the MLP beat the linear SVM beyond seed-to-seed variance (compare mean and std)? On a 569-sample dataset with well-separated classes, a tie is the expected result and is a legitimate finding.
- Clinical framing: a false negative (missed malignancy) is costlier than a false positive, so look at sensitivity first.